# Premier League fixtures - seasons 2023 to 2026

Fetches one complete API-Football response for each season (2026, 2025, 2024, and 2023 by default) and saves each untransformed response as a timestamped JSON snapshot under its corresponding `data/bronze/fixtures/league_id=39/season=YYYY/` folder. Set `FIXTURE_SEASONS` to a comma-separated list to select seasons for a run. Set `BRONZE_DATA_ROOT` to an absolute local bronze root when running on an agent. This endpoint currently returns all 380 fixtures for a season in one response; adding a `page` parameter causes an API error.

Before running, make `API_FOOTBALL_KEY` available to the notebook process. A key set in your computer's environment is available when running locally; a Databricks job needs the key configured in its own environment. Run with the repository as the working directory (or from a subdirectory of it).

In [ ]:
import json
import os
import shutil
from datetime import datetime, timezone
from pathlib import Path

import requests

API_URL = "https://v3.football.api-sports.io/fixtures"
LEAGUE_ID = 39  # Premier League
default_seasons = (2026, 2025, 2024, 2023)
configured_seasons = os.environ.get("FIXTURE_SEASONS")
if configured_seasons is None:
    SEASONS = default_seasons
else:
    try:
        SEASONS = tuple(int(value.strip()) for value in configured_seasons.split(","))
    except ValueError as error:
        raise ValueError("FIXTURE_SEASONS must be a comma-separated list of season years.") from error
    if not SEASONS or any(season <= 0 for season in SEASONS):
        raise ValueError("FIXTURE_SEASONS must contain positive season years.")
    if len(SEASONS) != len(set(SEASONS)):
        raise ValueError("FIXTURE_SEASONS must not contain duplicate season years.")

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / ".git").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError(
        "Could not find the repository root. Start the notebook with the Git repository as the working directory."
    )

configured_bronze_root = os.environ.get("BRONZE_DATA_ROOT")
bronze_root = Path(configured_bronze_root).expanduser() if configured_bronze_root else repo_root / "data" / "bronze"
if not bronze_root.is_absolute():
    raise ValueError("BRONZE_DATA_ROOT must be an absolute path when set.")

bronze_league_root = (
    bronze_root
    / "fixtures"
    / f"league_id={LEAGUE_ID}"
)
print(f"Bronze output directory: {bronze_league_root}")

In [ ]:
api_key = os.environ.get("API_FOOTBALL_KEY")
if not api_key:
    raise RuntimeError(
        "API_FOOTBALL_KEY is not set in this notebook process. Configure it in the environment and rerun."
    )

headers = {"x-apisports-key": api_key}
season_payloads = {}

for season in SEASONS:
    response = requests.get(
        API_URL,
        headers=headers,
        params={"league": LEAGUE_ID, "season": season},
        timeout=60,
    )
    response.raise_for_status()
    payload = response.json()

    if payload.get("errors"):
        raise RuntimeError(f"API-Football returned errors for season {season}: {payload['errors']}")
    if not isinstance(payload.get("response"), list):
        raise ValueError(f"API-Football response is missing its fixture list for season {season}.")

    parameters = payload.get("parameters")
    if not isinstance(parameters, dict):
        raise ValueError(f"API-Football response is missing request parameters for season {season}.")
    if parameters.get("league") != str(LEAGUE_ID) or parameters.get("season") != str(season):
        raise ValueError(f"API-Football response parameters do not match league {LEAGUE_ID}, season {season}.")

    paging = payload.get("paging")
    if not isinstance(paging, dict):
        raise ValueError(f"API-Football response is missing pagination metadata for season {season}.")
    if paging.get("current") != 1 or paging.get("total") != 1:
        raise ValueError(
            f"Expected a single complete fixtures response for season {season}. "
            "Review the API paging metadata before ingesting."
        )

    season_payloads[season] = payload
    print(f"Fetched {len(payload['response'])} fixtures for season {season}.")

In [ ]:
for season, payload in season_payloads.items():
    bronze_root = bronze_league_root / f"season={season}"
    bronze_root.mkdir(parents=True, exist_ok=True)
    ingested_at = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    snapshot_dir = bronze_root / f"ingested_at={ingested_at}"
    staging_dir = bronze_root / f".tmp-{ingested_at}"
    staging_dir.mkdir()

    try:
        page_path = staging_dir / "page_1.json"
        with page_path.open("w", encoding="utf-8") as output_file:
            json.dump(payload, output_file, ensure_ascii=False, indent=2)
            output_file.write("\n")
        staging_dir.rename(snapshot_dir)
    finally:
        if staging_dir.exists():
            shutil.rmtree(staging_dir)

    print(f"Saved season {season} response to {snapshot_dir}")